# 04 – Feature Engineering
Demonstrates all engineered features: temporal, weather, road infrastructure, encoding, and target creation.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.build_features import (
    temporal_features, weather_features, road_features,
    encode_categoricals, build_target, select_features
)
sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
df = pd.read_parquet('../data/interim/accidents_cleaned.parquet')
print(f'Loaded: {df.shape}')

## Temporal Features

In [ ]:
df = temporal_features(df)
print('New temporal columns:', ['hour','day_of_week','month','year','is_weekend','is_rush_hour','is_night','season','duration_min'])
print(df[['hour','day_of_week','month','year','is_weekend','is_rush_hour','duration_min']].describe().round(2))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
df.groupby('hour')['Severity'].mean().plot(ax=axes[0,0], marker='o', color='steelblue')
axes[0,0].set_title('Avg Severity by Hour')
df.groupby('day_of_week')['Severity'].mean().plot(ax=axes[0,1], kind='bar', color='coral')
axes[0,1].set_title('Avg Severity by Day of Week')
df.groupby('month')['Severity'].mean().plot(ax=axes[0,2], marker='o', color='green')
axes[0,2].set_title('Avg Severity by Month')
df.groupby('season')['Severity'].mean().plot(ax=axes[1,0], kind='bar', color='purple')
axes[1,0].set_title('Avg Severity by Season (0=Winter)')
df.groupby('is_rush_hour')['Severity'].mean().plot(ax=axes[1,1], kind='bar', color='orange')
axes[1,1].set_title('Avg Severity: Rush Hour vs Not')
df['duration_min'].clip(0, 120).hist(ax=axes[1,2], bins=50, color='teal', edgecolor='white')
axes[1,2].set_title('Accident Duration (min, capped 120)')
plt.suptitle('Temporal Feature Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Weather Features

In [ ]:
df = weather_features(df)
print('Weather severity score distribution:')
print(df['weather_severity_score'].value_counts().sort_index())
if 'weather_category' in df.columns:
    print('\nWeather categories:')
    print(df['weather_category'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.groupby('weather_severity_score')['Severity'].mean().plot(ax=axes[0], kind='bar', color='tomato')
axes[0].set_title('Avg Accident Severity by Weather Score')
if 'weather_category' in df.columns:
    df.groupby('weather_category')['Severity'].mean().sort_values().plot(ax=axes[1], kind='barh', color='steelblue')
    axes[1].set_title('Avg Severity by Weather Category')
plt.tight_layout()
plt.show()

## Road Infrastructure Features

In [ ]:
df = road_features(df)
print('Road feature count distribution:')
print(df['road_feature_count'].value_counts().sort_index())
fig, ax = plt.subplots(figsize=(8, 4))
df.groupby('road_feature_count')['Severity'].mean().plot(ax=ax, kind='bar', color='mediumseagreen')
ax.set_title('Avg Severity by Road Feature Count')
ax.set_xlabel('Number of Road Features Present')
plt.tight_layout()
plt.show()

## Target Variable

In [ ]:
df = build_target(df, threshold=3)
print('Binary target distribution:')
print(df['target'].value_counts())
print(f'\nClass imbalance ratio: {df["target"].value_counts()[0] / df["target"].value_counts()[1]:.2f}:1')
fig, ax = plt.subplots(figsize=(6, 4))
df['target'].value_counts().plot(kind='bar', ax=ax, color=['steelblue', 'tomato'])
ax.set_xticklabels(['Low (0)', 'High (1)'], rotation=0)
ax.set_title('Binary Target Distribution')
plt.tight_layout()
plt.show()

## Feature Selection Summary

In [ ]:
df, le_map = encode_categoricals(df)
features = select_features(df)
print(f'Total features selected: {len(features)}')
print('\nFeature list:')
for i, f in enumerate(features, 1):
    print(f'  {i:2d}. {f}')

In [ ]:
# Feature-target correlation
corr_with_target = df[features + ['target']].corr()['target'].drop('target').sort_values(key=abs, ascending=False)
fig, ax = plt.subplots(figsize=(10, 6))
corr_with_target.head(20).plot(kind='barh', ax=ax, color=['tomato' if v > 0 else 'steelblue' for v in corr_with_target.head(20)])
ax.set_title('Top 20 Feature Correlations with Target')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()